In [5]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Load the cleaned dataset: target (price_usd) + 10 approved features + game_id for grouping
model_input_v1 = pd.read_csv('/content/drive/MyDrive/final_model_output_small.csv')

In [8]:
feature_cols = [
    'days_out_approx',
    'platform_SeatGeek', 'platform_StubHub', 'platform_TickPick', 'platform_TicketIQ',
    'home_qb_repeat_appearance', 'away_qb_repeat_appearance',
    'home_team_appearance_no', 'away_team_appearance_no',
    'spread_line'
]

# x = features; force T/F to 1/0
X = model_input_v1[feature_cols].astype(float)
# y = target
y = model_input_v1['price_usd']

# set up logo id
groups = model_input_v1['game_id']

In [9]:
# LOGO-CV setup
logo = LeaveOneGroupOut()
fold_results = []

In [10]:
# Cross Validation Loop
for train_idx, test_idx in logo.split(X, y, groups):
    # Splitting the data - leave one group out; 9 games total
    # 8 train; 1 test (train/test split)
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    held_out_game = groups.iloc[test_idx].iloc[0]

    # Build a fresh model
    # Intentionally shallow since there isn't much data to learn from
    # Are these parameters good? (hyperparameter tuning)
    model = GradientBoostingRegressor(
        n_estimators=50,
        max_depth=2,
        min_samples_leaf=3,
        random_state=42
    )

    # Train model (one logo round)
    model.fit(X_train, y_train)
    # Model prediction
    preds = model.predict(X_test)

    # Model evaluation:
    # root mean square error; penalizes big misses more
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    # mean absolute error
    mae = mean_absolute_error(y_test, preds)

    # save all logo results
    fold_results.append({
        'held_out_game': held_out_game,
        'n_test_rows': len(test_idx),
        'rmse': rmse,
        'mae': mae
    })

In [11]:
# Print all 9 round scores into one table; observe per-game performance
results_df = pd.DataFrame(fold_results)
print(results_df)

      held_out_game  n_test_rows         rmse          mae
0    Super Bowl LII            1  3924.922099  3924.922099
1   Super Bowl LIII            3  5267.609559  5214.858578
2    Super Bowl LIV            2  1378.275323  1376.786983
3    Super Bowl LIX            2  3183.667415  2972.245737
4     Super Bowl LV            2  4079.883964  3582.661944
5    Super Bowl LVI            4  1039.521074   999.696032
6   Super Bowl LVII            1  2419.327167  2419.327167
7  Super Bowl LVIII            4  3805.087668  3504.291433
8     Super Bowl LX            2  1109.508388  1109.500000


In [12]:
# Overall picture
# Avg error across all 9 games & how much that error
# Swings from game to game (a big swing = model is inconsistent across games)

print(f"\nMean RMSE: {results_df['rmse'].mean():.2f} (± {results_df['rmse'].std():.2f})")
print(f"Mean MAE:  {results_df['mae'].mean():.2f} (± {results_df['mae'].std():.2f})")


Mean RMSE: 2911.98 (± 1506.67)
Mean MAE:  2789.37 (± 1436.30)


Honest read: Given ticket prices in your data run roughly $4,000–$12,000, a typical miss of ~$2,800 is a substantial chunk of the price itself — this isn't "tight and reliable" yet. And the model clearly does much better on some games than others, which is a fair thing to say plainly rather than gloss over.

Why this might be happening (not certain, just plausible): with only ~19 rows to train on per round and 10 features, the model doesn't have much to learn general patterns from — some games (LIII, LV) might just be genuinely harder to predict from the others, or there may be a feature or two adding noise rather than signal.

In [13]:
# Try 2: Shallower trees, bigger leaf requirement

fold_results_v2 = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    held_out_game = groups.iloc[test_idx].iloc[0]

    model = GradientBoostingRegressor(
        n_estimators=50,
        max_depth=1,          # shallower: each tree can only split once
        min_samples_leaf=5,   # bigger minimum group size per leaf
        random_state=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    fold_results_v2.append({
        'held_out_game': held_out_game,
        'n_test_rows': len(test_idx),
        'rmse': rmse,
        'mae': mae
    })

In [16]:
results_df_v2 = pd.DataFrame(fold_results_v2)
print("V2 - Shallower + bigger leaves (max_depth=1, min_samples_leaf=5)")
print(results_df_v2)
print(f"Mean RMSE: {results_df_v2['rmse'].mean():.2f} (± {results_df_v2['rmse'].std():.2f})")
print(f"Mean MAE:  {results_df_v2['mae'].mean():.2f} (± {results_df_v2['mae'].std():.2f})\n")

V2 - Shallower + bigger leaves (max_depth=1, min_samples_leaf=5)
      held_out_game  n_test_rows         rmse          mae
0    Super Bowl LII            1  3991.630234  3991.630234
1   Super Bowl LIII            3  3023.114818  3016.323120
2    Super Bowl LIV            2  1155.899676  1155.899007
3    Super Bowl LIX            2  2607.999583  2406.995957
4     Super Bowl LV            2  3860.191051  3481.678194
5    Super Bowl LVI            4  1592.593164  1406.982151
6   Super Bowl LVII            1  2909.204302  2909.204302
7  Super Bowl LVIII            4  4857.432631  4577.406581
8     Super Bowl LX            2  1056.012946  1043.533623
Mean RMSE: 2783.79 (± 1326.44)
Mean MAE:  2665.52 (± 1266.05)



Slightly better on both counts:

V1 (original)	V2 (shallower + bigger leaves)
Mean RMSE	$2,911.98	$2,783.79
Mean MAE	$2,789.37	$2,665.52
RMSE std (± spread)	$1,506.67	$1,326.44

Mean error dropped by about $125-130, and the spread across games tightened too (± came down from ~$1,507 to ~$1,326) — meaning V2 isn't just a bit more accurate on average, it's also a bit more consistent game-to-game. That's a real, if modest, improvement.

Worth noting honestly: it's not a dramatic fix. Super Bowl LVIII actually got worse under V2 ($3,805 → $4,857), while LIII, LIX, and LVII improved. So V2 traded a bit of accuracy on one game for gains on others — net positive, but not a clean win everywhere.

In [19]:
# Try 3: Fewer trees, same shallow depth as V2
fold_results_v3 = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    held_out_game = groups.iloc[test_idx].iloc[0]

    model = GradientBoostingRegressor(
        n_estimators=15,       # fewer trees than V1/V2
        max_depth=1,
        min_samples_leaf=5,
        random_state=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    fold_results_v3.append({
        'held_out_game': held_out_game,
        'n_test_rows': len(test_idx),
        'rmse': rmse,
        'mae': mae
    })

results_df_v3 = pd.DataFrame(fold_results_v3)
print("V3 - Fewer trees + shallower (n_estimators=15, max_depth=1, min_samples_leaf=5)")
print(results_df_v3)
print(f"Mean RMSE: {results_df_v3['rmse'].mean():.2f} (± {results_df_v3['rmse'].std():.2f})")
print(f"Mean MAE:  {results_df_v3['mae'].mean():.2f} (± {results_df_v3['mae'].std():.2f})\n")

V3 - Fewer trees + shallower (n_estimators=15, max_depth=1, min_samples_leaf=5)
      held_out_game  n_test_rows         rmse          mae
0    Super Bowl LII            1  3475.635117  3475.635117
1   Super Bowl LIII            3  3767.858455  3763.705702
2    Super Bowl LIV            2  1081.423951  1080.250000
3    Super Bowl LIX            2  2787.310714  2710.083621
4     Super Bowl LV            2  4060.224927  3861.314563
5    Super Bowl LVI            4  1914.141656  1726.542401
6   Super Bowl LVII            1  1721.613453  1721.613453
7  Super Bowl LVIII            4  4312.841270  3994.821532
8     Super Bowl LX            2  1121.172899  1109.500000
Mean RMSE: 2693.58 (± 1269.25)
Mean MAE:  2604.83 (± 1211.15)



V3 vs. baseline (V1):

V1 (original)	V3 (final tuned)
Mean RMSE	$2,911.98	$2,693.58
Mean MAE	$2,789.37	$2,604.83
RMSE spread (±)	$1,506.67	$1,269.25

Tuning shrunk the average error by ~$220 (RMSE) and tightened the game-to-game spread by ~$240 — a real, if modest, improvement from depth/leaf-size/tree-count adjustments alone.

Noted for future work: a Ridge regression comparison was considered as a simpler alternative model type, but not run for this version — flagged as a next step rather than pursued here.

In [20]:
# Train the final model on ALL available data (no holdout)
# This is the actual deployable model — not used for evaluation anymore,
# since evaluation already happened during LOGO-CV above
final_model = GradientBoostingRegressor(
    n_estimators=15,
    max_depth=1,
    min_samples_leaf=5,
    random_state=42
)
final_model.fit(X, y)

# Feature importance: which inputs the model leaned on most
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

                     feature  importance
5  home_qb_repeat_appearance    0.499395
9                spread_line    0.284427
6  away_qb_repeat_appearance    0.113451
2           platform_StubHub    0.102726
1          platform_SeatGeek    0.000000
0            days_out_approx    0.000000
4          platform_TicketIQ    0.000000
3          platform_TickPick    0.000000
7    home_team_appearance_no    0.000000
8    away_team_appearance_no    0.000000


In [21]:
# Save the trained model so it can be reloaded later without retraining
import joblib
joblib.dump(final_model, '/content/drive/MyDrive/final_model_v1.pkl')
print("Model saved.")

Model saved.
